# Evaluate SFT Models on GSM8K

This notebook evaluates fine-tuned Qwen models on the GSM8K test set.

**Hardware Requirements:**
- GPU: T4 or better (A100 recommended for faster inference)
- Runtime: Set to GPU in Colab

## 1. Setup and Installation

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q datasets

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 2. Import Libraries

In [ ]:
import os
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
import re
from tqdm.auto import tqdm
import json
import transformers.activations

# Patch for compatibility
if not hasattr(transformers.activations, "PytorchGELUTanh"):
    transformers.activations.PytorchGELUTanh = transformers.activations.GELUTanh

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Configuration

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

# Path to your fine-tuned model (on Google Drive or local)
MODEL_PATH = "/content/drive/MyDrive/agent_distillation/Qwen2.5-0.5B-Instruct_sft_merged"

# Evaluation settings
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True  # Set to False if you want full precision
NUM_TEST_SAMPLES = 100  # Number of samples to evaluate (GSM8K test has 1319 total)
MAX_NEW_TOKENS = 512  # Max tokens to generate
TEMPERATURE = 0.1  # Lower = more deterministic
TOP_P = 0.9

# Output settings
SAVE_PREDICTIONS = True
PREDICTIONS_FILE = "eval_predictions.jsonl"
VERBOSE = True  # Show first few examples

print(f"Configuration loaded!")
print(f"Model path: {MODEL_PATH}")
print(f"Evaluating on {NUM_TEST_SAMPLES} samples")

## 4. Load Model

In [ ]:
print(f"Loading model from: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
)

# Enable inference mode for faster generation
FastLanguageModel.for_inference(model)

print("Model loaded successfully!")

## 5. Load GSM8K Test Set

In [ ]:
print(f"Loading GSM8K test set...")

# Load full test set and optionally subsample
full_test_set = load_dataset("openai/gsm8k", "main", split="test")

if NUM_TEST_SAMPLES < len(full_test_set):
    test_dataset = full_test_set.select(range(NUM_TEST_SAMPLES))
else:
    test_dataset = full_test_set

print(f"Loaded {len(test_dataset)} test samples")
print(f"\nSample question:")
print(test_dataset[0]['question'])

## 6. Define Answer Extraction Function

In [ ]:
def extract_answer(text):
    """
    Extract numerical answer from model output.
    
    Tries multiple patterns:
    1. \boxed{number}
    2. "The answer is: number"
    3. Last number in text (fallback)
    """
    # 1. Try to find numbers inside \boxed{...}
    boxed_matches = re.findall(r'\\boxed\{([\d,]+)\}', text)
    if boxed_matches:
        return boxed_matches[-1].replace(",", "")
    
    # 2. Try the "The answer is: " format (from golden trajectories)
    matches = re.findall(r'[Tt]he answer is:?\s*([\d,]+)', text)
    if matches:
        return matches[-1].replace(",", "")
    
    # 3. Fallback: grab the last number in the text
    all_numbers = re.findall(r'\d+', text.replace(",", ""))
    return all_numbers[-1] if all_numbers else None

# Test the function
test_cases = [
    "The calculation shows that the answer is: 42",
    "Therefore, the result is \\boxed{100}",
    "After computing, we get 25 as the final answer"
]

print("Testing answer extraction:")
for test in test_cases:
    print(f"  '{test}' -> {extract_answer(test)}")

## 7. Run Evaluation

In [ ]:
print("="*60)
print("STARTING EVALUATION")
print("="*60)

correct = 0
predictions = []

# Progress bar
for i, example in enumerate(tqdm(test_dataset, desc="Evaluating")):
    # Format the question
    messages = [{"role": "user", "content": example['question']}]
    
    # Apply chat template and tokenize
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True if TEMPERATURE > 0 else False,
        )
    
    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract answer
    prediction = extract_answer(response)
    ground_truth = example['answer'].split("####")[-1].strip()
    
    # Check correctness
    is_correct = (prediction == ground_truth)
    if is_correct:
        correct += 1
    
    # Store prediction
    predictions.append({
        "question": example['question'],
        "ground_truth": ground_truth,
        "prediction": prediction,
        "response": response,
        "correct": is_correct
    })
    
    # Show first few examples
    if VERBOSE and i < 5:
        print(f"\n{'='*60}")
        print(f"Example {i+1}:")
        print(f"Question: {example['question']}")
        print(f"\nModel Response:\n{response}")
        print(f"\nExtracted: {prediction} | Ground Truth: {ground_truth} | Correct: {is_correct}")
        print(f"{'='*60}")
    
    # Periodic progress updates
    if (i + 1) % 20 == 0:
        current_acc = (correct / (i + 1)) * 100
        print(f"\n[Progress] {i+1}/{len(test_dataset)} | Accuracy: {current_acc:.2f}%")

# Final results
final_accuracy = (correct / len(test_dataset)) * 100

print("\n" + "="*60)
print("EVALUATION COMPLETE")
print("="*60)
print(f"Correct: {correct}/{len(test_dataset)}")
print(f"Accuracy: {final_accuracy:.2f}%")
print("="*60)

## 8. Save Results

In [ ]:
if SAVE_PREDICTIONS:
    # Save to local
    with open(PREDICTIONS_FILE, 'w') as f:
        for pred in predictions:
            f.write(json.dumps(pred) + '\n')
    
    print(f"Predictions saved to: {PREDICTIONS_FILE}")
    
    # Optionally save to Google Drive
    import shutil
    drive_path = f"/content/drive/MyDrive/agent_distillation/eval_results/{PREDICTIONS_FILE}"
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    shutil.copy(PREDICTIONS_FILE, drive_path)
    print(f"Predictions also saved to Drive: {drive_path}")

## 9. Error Analysis

In [ ]:
# Analyze errors
errors = [p for p in predictions if not p['correct']]

print(f"\nError Analysis:")
print(f"Total errors: {len(errors)}")
print(f"\nShowing first 3 errors:\n")

for i, error in enumerate(errors[:3]):
    print(f"{'='*60}")
    print(f"Error {i+1}:")
    print(f"Question: {error['question']}")
    print(f"\nGround Truth: {error['ground_truth']}")
    print(f"Prediction: {error['prediction']}")
    print(f"\nFull Response:\n{error['response'][:500]}...")
    print(f"{'='*60}\n")

## 10. Compare with Baseline (Zero-Shot CoT)

In [ ]:
# Optional: Evaluate base model without fine-tuning for comparison
# Uncomment to run baseline evaluation

# print("Loading base model for comparison...")
# base_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# base_model, base_tokenizer = FastLanguageModel.from_pretrained(
#     model_name=base_model_name,
#     max_seq_length=MAX_SEQ_LENGTH,
#     load_in_4bit=LOAD_IN_4BIT,
# )
# FastLanguageModel.for_inference(base_model)
# 
# base_correct = 0
# 
# for i, example in enumerate(tqdm(test_dataset.select(range(50)), desc="Baseline eval")):
#     # Add CoT prompting
#     cot_prompt = f"{example['question']}\n\nLet's think step by step."
#     messages = [{"role": "user", "content": cot_prompt}]
#     
#     input_ids = base_tokenizer.apply_chat_template(
#         messages,
#         add_generation_prompt=True,
#         return_tensors="pt"
#     ).to("cuda")
#     
#     with torch.no_grad():
#         outputs = base_model.generate(input_ids, max_new_tokens=MAX_NEW_TOKENS, temperature=0.1)
#     
#     response = base_tokenizer.decode(outputs[0], skip_special_tokens=True)
#     prediction = extract_answer(response)
#     ground_truth = example['answer'].split("####")[-1].strip()
#     
#     if prediction == ground_truth:
#         base_correct += 1
# 
# base_accuracy = (base_correct / 50) * 100
# print(f"\nBaseline (Zero-Shot CoT) Accuracy: {base_accuracy:.2f}%")
# print(f"Fine-tuned Model Accuracy: {final_accuracy:.2f}%")
# print(f"Improvement: {final_accuracy - base_accuracy:.2f} percentage points")

## 11. Summary and Next Steps

In [ ]:
print("\n" + "="*60)
print("EVALUATION SUMMARY")
print("="*60)
print(f"Model: {MODEL_PATH}")
print(f"Test samples: {len(test_dataset)}")
print(f"Correct predictions: {correct}")
print(f"Final Accuracy: {final_accuracy:.2f}%")
print("="*60)

print("\n📊 Performance Context:")
print("  - GPT-3.5-turbo (CoT): ~57% on GSM8K")
print("  - GPT-4: ~92% on GSM8K")
print("  - Qwen2.5-0.5B (base): ~10-15% on GSM8K")
print("  - Qwen2.5-7B (base): ~60-70% on GSM8K")

print("\n📝 Next Steps:")
print("1. If accuracy is good, proceed to GRPO optimization")
print("2. If accuracy is low, consider:")
print("   - Training for more steps")
print("   - Using a larger model (1.5B or 7B)")
print("   - Checking if golden trajectories are high quality")
print("3. Document baseline performance for paper")